In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle


In [2]:
## load data set
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15945405,Lewis,567,France,Male,58,0,13674.54,1,1,1,144022.31,0
1,2,15951000,Williams,571,Germany,Male,43,8,0.00,2,1,1,65399.77,0
2,3,15630085,Jones,628,Spain,Female,30,1,0.00,3,1,1,63760.04,0
3,4,15830429,Thomas,685,Spain,Female,44,8,107540.45,1,0,0,55274.68,1
4,5,15615278,King,737,France,Female,18,9,204248.69,2,0,0,176273.19,0


In [3]:
## preprocess the data
## dropping the irrelevant features

data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,567,France,Male,58,0,13674.54,1,1,1,144022.31,0
1,571,Germany,Male,43,8,0.00,2,1,1,65399.77,0
2,628,Spain,Female,30,1,0.00,3,1,1,63760.04,0
3,685,Spain,Female,44,8,107540.45,1,0,0,55274.68,1
4,737,France,Female,18,9,204248.69,2,0,0,176273.19,0
...,...,...,...,...,...,...,...,...,...,...,...
995,596,France,Female,54,8,215021.93,2,1,1,168623.76,0
996,747,Germany,Female,28,3,0.00,1,1,1,65839.12,0
997,848,France,Male,39,8,0.00,1,1,1,20811.28,0
998,499,Germany,Male,34,1,35779.44,2,1,1,19973.51,1


In [4]:
## Encode one categorical var
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,567,France,1,58,0,13674.54,1,1,1,144022.31,0
1,571,Germany,1,43,8,0.00,2,1,1,65399.77,0
2,628,Spain,0,30,1,0.00,3,1,1,63760.04,0
3,685,Spain,0,44,8,107540.45,1,0,0,55274.68,1
4,737,France,0,18,9,204248.69,2,0,0,176273.19,0
...,...,...,...,...,...,...,...,...,...,...,...
995,596,France,0,54,8,215021.93,2,1,1,168623.76,0
996,747,Germany,0,28,3,0.00,1,1,1,65839.12,0
997,848,France,1,39,8,0.00,1,1,1,20811.28,0
998,499,Germany,1,34,1,35779.44,2,1,1,19973.51,1


In [5]:
## OHE  Geography column

from sklearn.preprocessing import OneHotEncoder
ohe_geo = OneHotEncoder()
geo_encoder = ohe_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1000 stored elements and shape (1000, 3)>

In [6]:
geo_encoder.toarray() 

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(1000, 3))

In [7]:
ohe_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [8]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(),columns=ohe_geo.get_feature_names_out(['Geography']))

In [9]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,0.0,0.0,1.0
4,1.0,0.0,0.0
...,...,...,...
995,1.0,0.0,0.0
996,0.0,1.0,0.0
997,1.0,0.0,0.0
998,0.0,1.0,0.0


In [10]:
## Combine one hot encoder columns with the original data
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,567,1,58,0,13674.54,1,1,1,144022.31,0,1.0,0.0,0.0
1,571,1,43,8,0.00,2,1,1,65399.77,0,0.0,1.0,0.0
2,628,0,30,1,0.00,3,1,1,63760.04,0,0.0,0.0,1.0
3,685,0,44,8,107540.45,1,0,0,55274.68,1,0.0,0.0,1.0
4,737,0,18,9,204248.69,2,0,0,176273.19,0,1.0,0.0,0.0


In [11]:
## save the encoders and scaler
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('ohe_geo.pkl','wb') as file:
    pickle.dump(ohe_geo,file)

In [45]:
## Divide the dataset into independent and dependent features

X = data.drop('Exited',axis=1)
y = data['Exited']

## split the data in training and testing set

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, random_state=42)

## Scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
X_train

array([[ 1.21490937, -1.05395348,  0.38906395, ...,  1.00501256,
        -0.57542576, -0.58312392],
       [-0.13908739,  0.94880848,  1.18842487, ..., -0.99501244,
         1.73784364, -0.58312392],
       [-1.37199501,  0.94880848, -0.71005732, ..., -0.99501244,
        -0.57542576,  1.71490137],
       ...,
       [-1.16284104, -1.05395348,  1.98778579, ..., -0.99501244,
         1.73784364, -0.58312392],
       [ 0.55442315,  0.94880848,  0.0893036 , ..., -0.99501244,
        -0.57542576,  1.71490137],
       [ 0.36728538, -1.05395348, -0.71005732, ...,  1.00501256,
        -0.57542576, -0.58312392]], shape=(800, 12))

In [14]:
X_test

array([[ 0.22418003, -1.05395348,  1.58810533, ...,  1.00501256,
        -0.57542576, -0.58312392],
       [ 1.50112007,  0.94880848, -0.31037686, ..., -0.99501244,
         1.73784364, -0.58312392],
       [ 0.10309089, -1.05395348,  1.28834499, ...,  1.00501256,
        -0.57542576, -0.58312392],
       ...,
       [ 1.73229025,  0.94880848,  0.58890418, ...,  1.00501256,
        -0.57542576, -0.58312392],
       [-1.0417519 ,  0.94880848, -1.80917859, ...,  1.00501256,
        -0.57542576, -0.58312392],
       [ 0.52139884,  0.94880848,  0.28914383, ...,  1.00501256,
        -0.57542576, -0.58312392]], shape=(200, 12))

In [15]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler,file)
    

In [16]:
!pip install tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [17]:
(X_train.shape[1],) ## single dimension with 12 elements

(12,)

In [18]:
## Build Our ANN Model
model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), ## HL1 Connected with Input Layer
    Dense(32,activation='relu'), ## HL2 Connected with Input Layer
    Dense(1,activation='sigmoid'), ## Output  Layer
])

f:\PYTHON_AI\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [19]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)

In [20]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
## Compiler the model
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=['accuracy'])

In [34]:
## set up the tensorboard
!pip install TensorBoard
import tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [35]:
## Set up Early Stopping 
early_stop_cb = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [24]:
## Train the model 
history = model.fit(
    X_train,y_train,validation_data = (X_test,y_test),epochs=100,
    callbacks= (tensorflow_callback,early_stop_cb)
)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.6438 - loss: 0.6336 - val_accuracy: 0.7300 - val_loss: 0.5842
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7300 - loss: 0.5586 - val_accuracy: 0.7300 - val_loss: 0.5651
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7337 - loss: 0.5264 - val_accuracy: 0.7300 - val_loss: 0.5538
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7500 - loss: 0.5090 - val_accuracy: 0.7350 - val_loss: 0.5591
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7738 - loss: 0.4973 - val_accuracy: 0.7200 - val_loss: 0.5563
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7837 - loss: 0.4886 - val_accuracy: 0.7150 - val_loss: 0.5604
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7887 - loss: 0.4795 - val_accuracy: 0.7150 - val_loss: 0.5651
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7912 - loss: 0.4739 - val_accuracy: 0.

In [36]:
model.save('model.h5')
import sys
!{sys.executable} -m pip install tensorboard


In [41]:
## Load TensorBoard Extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [44]:
%tensorboard --logdir logs/fit20260927-191652

Reusing TensorBoard on port 6007 (pid 22668), started 0:06:59 ago. (Use '!kill 22668' to kill it.)